# 10 · PCSI source gate, candidate audit and ablations

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run after OOF training; candidate inference never uses the reference table.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Execute integrity and monotonicity tests

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_claims_and_gates.py')],cwd=REPO,check=True)

## 2. Generate validation and calibration candidates for independent review

In [ ]:
from oncoplate.inputs import make_partition_candidates
RUN_ID='dinov2_vits14_finetune_joint_s0'
context=read_json(p['private']/"protocol_context.json")
for partition in ['validation','calibration']:
    cc,objects,outdir=make_partition_candidates(cfg,RUN_ID,partition,input_mode=context['input_mode'],asof=context['asof'])
    display(cc[['record_id','field','value','qualifier','eligible','gate_reason']].head())
    print('Rating jobs:',outdir/f'{partition}_rating_jobs.csv')

## 3. Inspect controlled selector scores
No outcome column is supplied to the score function. These scores are uncalibrated until notebook 12.

In [ ]:
from oncoplate.evaluation import score_candidates
cc=read_table(outdir/'validation_candidates.csv')
scored=score_candidates(cc,p['private']/'selectors'/RUN_ID)
write_table(outdir/'validation_uncalibrated_selector_scores.csv',scored)
display(scored.groupby('gate_reason').size().rename('records'))

## 4. Record method scope
M3, M6 and M7 use exactly the same candidate. Hierarchical or VLM-generated alternatives need their own blind ratings; they are not silently treated as equivalent.

In [ ]:
print((REPO/'docs/BASELINES_AND_LIMITATIONS.md').read_text())
print('Next: 11 for development clarification episodes and 12 for calibration/analysis lock.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
